### Modules / Packages

In [ ]:
import os
from pathlib import Path
import json
import pandas as pd


PosixPath('/Users/riccardobertolini9/Desktop/project-teamX/data/raw/raw_credit_applications.json')

## **@ Riccardo** Some parts of the cleaning are missing. Please go through the checklist in the project overview document. Everything has to be included. Please use more markdown cells with explanations so I can understand better what you are doing. For the analysis we still need to fix the gender as there are "Female" and "Male" as well as "F" and "M". Also, you have to compute the age as an additional column. And don't always import pandas as pd again, you only have to do it once at the start (already done)

# Data Quality & Cleaning

This notebook performs data validation and cleaning of the credit applications dataset.
The goal is to ensure data consistency, handle missing values, and prepare the dataset for further bias analysis.

## 1. Initial Data Inspection

We inspect the dataset structure, number of records, and variable types to identify potential data quality issues.

In [1]:
# load json
with open("../data/raw_credit_applications.json") as f:
    data = json.load(f)

# normalize nested structure
df = pd.json_normalize(data)
df.shape

(502, 21)

In [2]:
df.head()

,_id,spending_behavior,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,...,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes
0,app_200,"[{'category': 'Shopping', 'amount': 480}, {'ca...",2024-01-15T00:00:00Z,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036,...,23,0.20,31212,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN
1,app_037,"[{'category': 'Rent', 'amount': 608}, {'catego...",NaN,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,M,1992-03-31,10032,...,51,0.18,17915,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN
2,app_215,"[{'category': 'Rent', 'amount': 109}]",NaN,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075,...,41,0.21,37909,True,NaN,vacation,3.7,59000.0,NaN,NaN
3,app_024,"[{'category': 'Fitness', 'amount': 575}]",NaN,Thomas Lee,thomas.lee6@protonmail.com,194-35-1833,192.168.175.67,Male,1983-04-25,10077,...,70,0.35,0,True,NaN,NaN,4.3,34000.0,NaN,NaN
4,app_184,"[{'category': 'Entertainment', 'amount': 463}]",2024-01-15T00:00:00Z,Brian Rodriguez,brian.rodriguez86@aol.com,480-41-2475,172.29.125.105,M,1999-05-21,10080,...,14,0.23,31763,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN


In [397]:
df.columns.tolist()[:10]

['_id',
 'spending_behavior',
 'processing_timestamp',
 'applicant_info.full_name',
 'applicant_info.email',
 'applicant_info.ssn',
 'applicant_info.ip_address',
 'applicant_info.gender',
 'applicant_info.date_of_birth',
 'applicant_info.zip_code']

In [398]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 502 entries, 0 to 501
Data columns (total 21 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   _id                               502 non-null    object 
 1   spending_behavior                 502 non-null    object 
 2   processing_timestamp              62 non-null     object 
 3   applicant_info.full_name          502 non-null    object 
 4   applicant_info.email              502 non-null    object 
 5   applicant_info.ssn                497 non-null    object 
 6   applicant_info.ip_address         497 non-null    object 
 7   applicant_info.gender             501 non-null    object 
 8   applicant_info.date_of_birth      501 non-null    object 
 9   applicant_info.zip_code           501 non-null    object 
 10  financials.annual_income          497 non-null    object 
 11  financials.credit_history_months  502 non-null    int64  
 12  financia

In [399]:
df.isnull().sum().sort_values(ascending=False)

notes                               500
financials.annual_salary            497
loan_purpose                        452
processing_timestamp                440
decision.rejection_reason           292
decision.approved_amount            210
decision.interest_rate              210
financials.annual_income              5
applicant_info.ip_address             5
applicant_info.ssn                    5
applicant_info.date_of_birth          1
applicant_info.zip_code               1
applicant_info.gender                 1
spending_behavior                     0
financials.credit_history_months      0
financials.debt_to_income             0
financials.savings_balance            0
decision.loan_approved                0
applicant_info.email                  0
applicant_info.full_name              0
_id                                   0
dtype: int64

## 2. Missing Values Analysis

We observe a high number of missing values in:
- `notes`
- `financials.annual_salary`
- `loan_purpose`
- `processing_timestamp`

These columns require special attention during the cleaning phase.
Columns with very few missing values may be handled with simple imputation or row removal.

## **@ Riccardo** what is the purpose of this part? There is no analysis.

## 3. Duplicate Check

In [400]:
# Check duplicates excluding nested column
df_no_nested = df.drop(columns=["spending_behavior"])

df_no_nested.duplicated().sum()

np.int64(0)

### Duplicate Analysis 

No duplicated records were found when excluding nested transaction data.
The nested column `spending_behavior` was excluded from the check due to its list/dictionary structure.

## 4. Handling Nested Transaction Data

The `spending_behavior` column contains a list of transactions for each applicant.  
To properly analyze spending patterns, we transform this nested structure into a tabular format.

The transformation involves:
1. Exploding the list into multiple rows
2. Expanding the dictionary into separate columns (category, amount)

## **@ Riccardo** please change the Italian comments to English

In [ ]:
# Partiamo da id + spending_behavior
sp = df[["_id", "spending_behavior"]].copy()

# Trasformiamo la lista in righe (explode)
sp = sp.explode("spending_behavior", ignore_index=True)

# Espandiamo i dict dentro spending_behavior in colonne (category, amount)
sp_details = pd.json_normalize(sp["spending_behavior"])
sp = pd.concat([sp.drop(columns=["spending_behavior"]), sp_details], axis=1)

sp.head(10)

,_id,category,amount
0,app_200,Shopping,480
1,app_200,Rent,790
2,app_200,Alcohol,247
3,app_037,Rent,608
4,app_037,Dining,96
5,app_037,Healthcare,243
6,app_215,Rent,109
7,app_024,Fitness,575
8,app_184,Entertainment,463
9,app_275,Entertainment,571


After transformation, each row represents a single transaction linked to an applicant ID.
This structure allows aggregation and further behavioral analysis.

In [402]:
sp.shape, sp.isna().mean().sort_values(ascending=False).head(10)

((827, 3),
 _id         0.0
 category    0.0
 amount      0.0
 dtype: float64)

In [403]:
# Aggregate spending per applicant
sp_agg = (
    sp.groupby("_id")
      .agg(
          total_spent=("amount", "sum"),
          avg_transaction=("amount", "mean"),
          num_transactions=("amount", "count")
      )
      .reset_index()
)

sp_agg.head()

,_id,total_spent,avg_transaction,num_transactions
0,app_001,1152,576.0,2
1,app_002,533,533.0,1
2,app_003,450,450.0,1
3,app_004,1139,569.5,2
4,app_005,585,585.0,1


### 4.1 Spending Aggregation

Transaction-level data is aggregated at applicant level to compute:
- Total spending
- Average transaction amount
- Number of transactions

This allows integration with the main application dataset.

In [404]:
sp_agg.shape

(500, 4)

### 4.2 Aggregation Validation

The aggregated dataset contains 500 unique applicants.
No applicant IDs were lost during the transformation process.
Transaction-level data has been successfully converted into applicant-level metrics.

In [405]:
missing_ids = set(df["_id"]) - set(sp_agg["_id"])
len(missing_ids), list(missing_ids)[:10]

(0, [])

In [406]:
missing_df = pd.DataFrame({
    "_id": list(missing_ids),
    "total_spent": 0,
    "avg_transaction": 0,
    "num_transactions": 0
})

In [407]:
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
missing_pct

notes                               99.601594
financials.annual_salary            99.003984
loan_purpose                        90.039841
processing_timestamp                87.649402
decision.rejection_reason           58.167331
decision.approved_amount            41.832669
decision.interest_rate              41.832669
financials.annual_income             0.996016
applicant_info.ip_address            0.996016
applicant_info.ssn                   0.996016
applicant_info.date_of_birth         0.199203
applicant_info.zip_code              0.199203
applicant_info.gender                0.199203
spending_behavior                    0.000000
financials.credit_history_months     0.000000
financials.debt_to_income            0.000000
financials.savings_balance           0.000000
decision.loan_approved               0.000000
applicant_info.email                 0.000000
applicant_info.full_name             0.000000
_id                                  0.000000
dtype: float64

In [408]:
sp_agg = pd.concat([sp_agg, missing_df], ignore_index=True)
sp_agg.shape

(500, 4)

## **@ Riccardo** Italian Comments

In [409]:
# quando loan NON è approvato
df[df["decision.loan_approved"] == False][
    ["decision.approved_amount", "decision.interest_rate"]
].isna().mean()

decision.approved_amount    1.0
decision.interest_rate      1.0
dtype: float64

In [410]:
# quando loan è approvato
df[df["decision.loan_approved"] == True][
    ["decision.approved_amount", "decision.interest_rate"]
].isna().mean()

decision.approved_amount    0.0
decision.interest_rate      0.0
dtype: float64

## 5. Consistency Check: Loan Approval Logic

When a loan is not approved:
- `approved_amount` and `interest_rate` are always missing.

When a loan is approved:
- These fields are always populated.

This confirms logical consistency between approval status and financial attributes.

In [411]:
(df["financials.credit_history_months"] < 0).sum()

np.int64(2)

In [412]:
((df["financials.debt_to_income"] < 0) | 
 (df["financials.debt_to_income"] > 1)).sum()

np.int64(1)

## **@ Riccardo** Please write more Markdown cells with explanation otherwise we cannot understand why you do that

In [413]:
# Use a fixed reference date to keep results reproducible
REF_DATE = pd.Timestamp("2024-12-31")

df["applicant_info.date_of_birth"] = pd.to_datetime(
    df["applicant_info.date_of_birth"], errors="coerce"
)

df["age"] = (REF_DATE - df["applicant_info.date_of_birth"]).dt.days / 365.25

In [414]:
(df["age"] < 18).sum(), (df["age"] > 100).sum()

(np.int64(0), np.int64(0))

In [415]:
df[df["financials.credit_history_months"] < 0][
    ["_id", "financials.credit_history_months"]
]

,_id,financials.credit_history_months
137,app_043,-10
162,app_156,-3


In [416]:
df[(df["financials.debt_to_income"] < 0) |
   (df["financials.debt_to_income"] > 1)][
    ["_id", "financials.debt_to_income"]
]

,_id,financials.debt_to_income
316,app_402,1.85


In [417]:
df.loc[df["financials.credit_history_months"] < 0,
       "financials.credit_history_months"] = pd.NA

In [418]:
(df["financials.credit_history_months"] < 0).sum()

np.int64(0)

In [419]:
df_apps = df.drop(columns=["spending_behavior"]).copy()
df_apps.shape

(502, 21)

## **@ Riccardo** Italian comments

In [ ]:
# df_apps deve esistere: se non l'hai creato, fallo ora
df_apps = df.drop(columns=["spending_behavior"]).copy()

bad_cols = []
for c in df_apps.columns:
    try:
        pd.DataFrame({c: df_apps[c]}).to_parquet("/tmp/test.parquet", index=False)
    except Exception as e:
        bad_cols.append((c, type(e).__name__, str(e)[:200]))

bad_cols

[('financials.annual_income',
  'ArrowInvalid',
  '("Could not convert \'55000\' with type str: tried to convert to int64", \'Conversion failed for column financials.annual_income with type object\')')]

In [421]:
# forziamo annual_income a numeric: stringhe -> numeri, valori non convertibili -> NaN
df_apps["financials.annual_income"] = pd.to_numeric(
    df_apps["financials.annual_income"],
    errors="coerce"
)

df_apps["financials.annual_income"].dtype

dtype('float64')

In [422]:
df_apps["financials.annual_income"].isna().sum()

np.int64(5)

In [423]:
df_apps.to_parquet("data/processed/credit_applications_clean.parquet", index=False)
sp.to_parquet("data/processed/spending_behavior.parquet", index=False)

In [424]:
cols_to_remove = [
    "applicant_info.ssn",
    "applicant_info.ip_address"
]

df_clean = df_apps.drop(columns=cols_to_remove).copy()

df_clean.shape

(502, 19)

In [425]:
missing_ratio = df_clean.isna().mean()

cols_high_missing = missing_ratio[missing_ratio > 0.9].index.tolist()

cols_high_missing

['loan_purpose', 'financials.annual_salary', 'notes']

In [426]:
df_clean = df_clean.drop(columns=cols_high_missing)
df_clean.shape

(502, 16)

In [427]:
df_clean.to_parquet(
    "data/clean/credit_applications_final.parquet",
    index=False
)

## **@ Riccardo** What are these functions for? When are we using them? And please keep the comments without dashes

In [ ]:
def clean_credit_applications(df_raw):
    
    df = pd.json_normalize(df_raw)
    
    # --- Remove nested column ---
    if "spending_behavior" in df.columns:
        df = df.drop(columns=["spending_behavior"])
    
    # --- Fix invalid values ---
    df.loc[df["financials.credit_history_months"] < 0,
           "financials.credit_history_months"] = pd.NA
    
    df.loc[
        (df["financials.debt_to_income"] < 0) |
        (df["financials.debt_to_income"] > 1),
        "financials.debt_to_income"
    ] = pd.NA
    
    # --- Fix type mismatch ---
    df["financials.annual_income"] = pd.to_numeric(
        df["financials.annual_income"],
        errors="coerce"
    )
    
    # --- Remove sensitive columns ---
    sensitive_cols = [
        "applicant_info.ssn",
        "applicant_info.ip_address"
    ]
    df = df.drop(columns=sensitive_cols, errors="ignore")
    
    # --- Remove high missing columns ---
    missing_ratio = df.isna().mean()
    high_missing = missing_ratio[missing_ratio > 0.9].index
    df = df.drop(columns=high_missing)
    
    return df

In [429]:
def clean_data(df):

    # --- Fix invalid debt_to_income ---
    df.loc[
        (df["financials.debt_to_income"] < 0) |
        (df["financials.debt_to_income"] > 1),
        "financials.debt_to_income"
    ] = pd.NA

    # --- Fix invalid credit history ---
    df.loc[
        df["financials.credit_history_months"] < 0,
        "financials.credit_history_months"
    ] = pd.NA

    # --- Fix income type mismatch ---
    df["financials.annual_income"] = pd.to_numeric(
        df["financials.annual_income"],
        errors="coerce"
    )

    # --- Remove sensitive columns ---
    sensitive_cols = [
        "applicant_info.ssn",
        "applicant_info.ip_address"
    ]
    df = df.drop(columns=sensitive_cols, errors="ignore")

    # --- Remove high missing columns ---
    missing_ratio = df.isna().mean()
    high_missing = missing_ratio[missing_ratio > 0.9].index
    df = df.drop(columns=high_missing)

    return df

In [430]:
df_clean = clean_data(df.copy())
df.shape, df_clean.shape

((502, 22), (502, 17))

In [ ]:
# Duplicate check
dup_id = df_clean["_id"].duplicated().sum()

dup_id

np.int64(2)

In [432]:
# Fix duplicated applicant IDs by keeping the first occurrence
df_clean = df_clean.sort_values("_id").drop_duplicates(subset=["_id"], keep="first")

# Verify duplicates are removed
df_clean["_id"].duplicated().sum()

np.int64(0)

In [433]:
df_clean["_id"].duplicated().sum()

np.int64(0)

In [434]:
df_clean[df_clean["_id"].duplicated(keep=False)].sort_values("_id")

,_id,spending_behavior,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,financials.annual_income,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,decision.interest_rate,decision.approved_amount,age


In [435]:
df_clean = df_clean.drop_duplicates(subset=["_id"], keep="first").copy()

df_clean["_id"].duplicated().sum()

np.int64(0)

## **@ Riccardo** Update the following part when finished

###  6. Data Quality Summary

The following issues were identified and addressed:

- 2 duplicate application IDs → resolved by keeping first occurrence.
- Invalid credit history values (<0) → replaced with NA.
- Debt-to-income ratios outside valid range [0,1] → replaced with NA.
- Annual income stored as string in some records → converted to numeric.
- Columns with >90% missing values removed.
- Sensitive personal identifiers (SSN, IP address) removed.

Final dataset:
- 502 records
- 17 columns
- No duplicate IDs
- Cleaned and exported to parquet format

In [438]:
print("Final dataset shape:", df_clean.shape)

print("\nMissing values per column:")
print(df_clean.isnull().sum())

print("\nDuplicate application IDs:",
      df_clean["_id"].duplicated().sum())

print("\nBasic statistics:")
print(df_clean.describe())

Final dataset shape: (500, 17)

Missing values per column:
_id                                   0
spending_behavior                     0
processing_timestamp                438
applicant_info.full_name              0
applicant_info.email                  0
applicant_info.gender                 1
applicant_info.date_of_birth        162
applicant_info.zip_code               1
financials.annual_income              5
financials.credit_history_months      2
financials.debt_to_income             1
financials.savings_balance            0
decision.loan_approved                0
decision.rejection_reason           292
decision.interest_rate              208
decision.approved_amount            208
age                                 162
dtype: int64

Duplicate application IDs: 0

Basic statistics:
        applicant_info.date_of_birth  financials.annual_income  \
count                            338                495.000000   
mean   1984-08-21 11:55:44.378698240              82693.803614   
m

In [439]:
df_clean = df_clean.drop(columns=["processing_timestamp"], errors="ignore")

In [440]:
df_clean = df_clean.sort_values("_id").drop_duplicates(subset=["_id"], keep="first")

In [ ]:
df_clean.to_csv(
    "data/credit_applications_clean.csv",
    index=False
)